**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/semenov-2026-001518"], check=True)
    os.chdir("/content/encore/entries/semenov-2026-001518")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Semenov et al. (2026) on dandiset 001518

**Paper.** Semenov I, Silkuniene G, Silkunas M, Pakhomova ON, Bixler JN, Ibey BL, Kochunov P,
Pakhomov AG (2026). Sub-microsecond optical measurements of cell membrane charging and lesioning
by pulsed electric fields. Bioelectrochemistry 171, 109283. doi:10.1016/j.bioelechem.2026.109283.

**Dataset.** DANDI:001518, version 0.261006.1957
(doi:10.48324/dandi.001518/0.261006.1957), streamed from the archive. The paper cites no
version of the dataset. This is the only published version and the most recent one, so it
is pinned (protocol, author step 1.3). The draft has the same 190 assets.

**Claims.** `claims/001518/10.1016-j.bioelechem.2026.109283.yaml`, written from the version of
record (main text, Methods, Results, Discussion and figure legends; no supplement).

**Outside input.** The charging time constant of Eq. (1) is taken from reference [20] of the
paper, Semenov et al. (2025), Bioelectrochemistry 168, 109143,
doi:10.1016/j.bioelechem.2025.109143: tau = 44.5 ns/um x R, the fit through the origin in its
Results. The paper says it used this fit (Methods 2.3).

**Revision after verification round 1.** The factor from volts to V/cm is now taken as 5.6 V/cm
per volt (`top`, 1400 V/cm at 250 V), which agrees with the electrode geometry of Methods 2.2.
The earlier choice (`bottom`, 1.72) was fitted to the dyes' calibration, which
fig3-dye-sensitivity tests. The Di-8-ANEPPS claims at 570 V/cm and above are therefore tested.
Added: a check of the automated trace with an edge-following trace as a variant, a
camera-offset background reading, pole values at all
19 sets, a drift control for fig2b-di8-decline-within-set, an offset-removed variant for
fig3-fluovolt-nonlinear, and every criterion over a sweep of factors (section before the summary).

**Revision after verification round 2.** No outcome changed. Added a fine grid of factors (1.40
to 2.30 V/cm per volt) for fig3-dye-sensitivity, and corrected the baseline range in
fig2a-fluovolt-stable and the reading of the bleaching offset in fig7b-charging-during-pulse.

Claims and how each is treated (outcomes as in the summary):

| Claim | Tested | Outcome |
|---|---|---|
| fig3-dye-sensitivity | tested | not reproduced (assumed factor) |
| fig3-fluovolt-nonlinear | tested | not reproduced (assumed factor) |
| fig2a-fluovolt-saturation | tested | not reproduced |
| fig2a-fluovolt-stable | tested | not reproduced (assumed factor) |
| fig4a-fluovolt-limits | tested | not reproduced |
| fig4a-fluovolt-cathodic-decline | tested | not reproduced |
| fig2b-di8-climbs-to-570 | tested | partially reproduced (assumed factor) |
| fig2b-di8-decline-within-set | tested | reproduced (assumed factor and pole) |
| fig4b-di8-limits | tested | not reproduced |
| fig4b-first-10-pulses | tested | not reproduced |
| fig4b-cathodic-asymmetry | tested | not reproduced (assumed factor) |
| fig4c-equator-follows-cosine | tested | not reproduced (assumed factor) |
| fig1-pole-dimming | tested | not reproduced (assumed factor) |
| fig7b-charging-during-pulse | tested | not reproduced (assumed factor) |
| fig5b-conductivity-calibration | tested (Eq. 2, no recordings) | partially reproduced |
| fig2cd-stable-before-electroporation, fig2c-1000-within-set, fig2d-lasting-reduction | not in the dataset (alternating protocol) | blocked: data missing |
| fig4d-short-pulses-same-limit | not in the dataset (0.3-us pulses) | blocked: data missing |
| fig5a-gd-extends-anodic, fig5a-gd-linear, fig5a-gd-cathodic, fig5a-gd-differences, fig5c-gd-reduces-conductivity | not in the dataset (Gd3+) | blocked: data missing |
| fig6a-ca-single-pulse, fig6b-ca-train-responders, fig6b-ca-650 | not in the dataset (Fluo-4 Ca2+ imaging) | blocked: data missing |
| fig7ab-test-pulse-recovery, fig7c-fast-conductivity-drop, fig7c-full-reseal-mild | not in the dataset (two-pulse protocol) | blocked: data missing |

Outcomes marked "assumed factor" rest on the factor from volts to V/cm. The others fail under
every factor from 1.5 to 8 V/cm per volt. All tested outcomes but fig5b-conductivity-calibration rest on the automated trace, the pixel size and
which cells stand for the paper's one cell per experiment.

In [ ]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess

import pef

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("dandiset", pef.DANDISET, "version", pef.VERSION)

In [ ]:
# Streams all 190 files (about 7 GB) and keeps only the membrane profiles; about 10 minutes.
started = time.time()
sets = pef.load_all()

In [ ]:
tracked = pef.track(sets)
resp = pef.responding(tracked)
OFFSET = pef.camera_offset(sets)
D = pef.pole_rows(resp)  # background subtracted, reading (b)
D_raw = pef.pole_rows(resp, background=False)  # no background subtraction, reading (a)
D_off = pef.pole_rows(resp, background=OFFSET)  # camera offset subtracted only, a variant
D_edge = pef.pole_rows(pef.with_trace(resp, "edge"))  # edge trace (helper, trace_edge), a variant
print(f"{len(sets)} sets loaded and reduced in {(time.time() - started) / 60:.1f} min")

## Data check

Each file is one set of 100 images. The session description gives the dye and the pulse
amplitude in volts; there is no table of trials, no cell outline, no field direction and no
pixel size. The checks below compare what the paper describes with the values.

In [ ]:
inv = pd.DataFrame([dict(subject=s["subject"], dye=s["dye"], volt=s["volt"], start=s["start"],
                         spacing_mm=s["spacing_mm"], n_frames=len(s["good"]), n_dark=int((~s["good"]).sum()),
                         dark_first=bool(~s["good"][0]), n_cells_found=len(s["cells"])) for s in sets])
print("Sets:", len(inv), " subjects:", inv.subject.nunique(), " frames per set:", sorted(inv.n_frames.unique()))
print("\nSets per subject and dye:")
print(inv.groupby(["dye", "subject"]).agg(sets=("volt", "size"), volts=("volt", lambda v: f"{v.min()}-{v.max()} V"),
                                           span_min=("start", lambda t: round((t.max() - t.min()) / 60, 1))))
print("\nVoltages (same 19 in every subject):", sorted(inv.volt.unique()))
print("Every subject has each voltage once:", bool((inv.groupby(["subject", "volt"]).size() == 1).all()))
order = inv.sort_values("start").groupby("subject").volt.apply(lambda v: bool(np.all(np.diff(v.values) > 0)))
print("Voltage increases set by set in every subject:", bool(order.all()))
gaps = inv.sort_values("start").groupby("subject").start.apply(lambda t: np.diff(t.values)).explode().astype(float)
print(f"Time between set starts: median {gaps.median():.0f} s, range {gaps.min():.0f} to {gaps.max():.0f} s")
print("Electrode distance stated in the files (mm):", sorted(inv.spacing_mm.unique()), "; paper, Methods 2.2: 1.6 mm")

The dandiset holds ten experiments from one day, five with Di-8-ANEPPS (c1 to c5) and five with
FluoVolt (c7 to c11). Each is one escalation of 19 sets of 1-us pulses from 25 to 250 V, with the
voltage raised set by set. The paper's escalations (Fig. 2A, 2B and 4A, 4B) have nine sets, 43 to
1400 V/cm (Fig. 2 legend: "9 cycles"), with n = 4 for FluoVolt and 5 for Di-8-ANEPPS. The files
say the electrodes are 1.00 mm apart; the paper says 1.6 mm. Nothing in the files records the
alternating protocol (Fig. 2C, D), 0.3-us pulses (Fig. 4D), Gd3+ (Fig. 5), Fluo-4 Ca2+ imaging
(Fig. 6) or the two-pulse protocol (Fig. 7).

The voltages span a factor of 10, while the paper's field strengths span a factor of 1400 / 43 =
33. Field strength is proportional to voltage (Methods 2.2), so no single factor maps all nine
of the paper's levels onto these sets. The factor from volts to V/cm is in neither the paper,
its reference [20], nor the files. Two endpoint readings are run: `top` puts the paper's highest
level on the highest voltage (1400 V/cm at 250 V, 5.6 V/cm per V), and `bottom` puts its lowest on
the lowest (43 V/cm at 25 V, 1.72 V/cm per V). The data check below weighs them.

In [ ]:
print("Dark frames (laser misfires): frame 0 is dark in", int(inv.dark_first.sum()), "of", len(inv), "sets;",
      int((inv.n_dark - inv.dark_first).sum()), "other dark frames in", int(((inv.n_dark - inv.dark_first) > 0).sum()), "sets")
fm = np.array([s["frame_mean"] for s in sets])
good = np.array([s["good"] for s in sets])
print(f"Whole-image mean: lit frames {np.median(fm[good]):.0f} (range {fm[good].min():.0f} to {fm[good].max():.0f}),"
      f" dark frames {np.median(fm[~good]):.0f} (range {fm[~good].min():.0f} to {fm[~good].max():.0f}) counts")
bg = np.concatenate([s["bg"][s["good"]] for s in sets])
print(f"Cell-free background in lit frames: median {np.median(bg):.0f}, range {bg.min():.0f} to {bg.max():.0f} counts (unit 'n.a.' in the files)")

Frame 0 of almost every set, and a few other frames, are dark: their mean is near the camera
offset, so the laser did not fire. They are left out (helper, `good_frames`); with frame 0 out,
the baseline has 39 images in most sets, not the paper's 40. The cell-free background is
about 330 counts, a large part of the membrane signal, so background subtraction about doubles
dF (ranges printed below); both readings are run where the claims file names them.

In [ ]:
cell_tab = pd.DataFrame([dict(subject=sub, dye=ss[0]["dye"], found_first_set=len(ss[0]["cells"]), in_every_set=n_full,
                              responding=len(keep), cathode_deg=round(cath))
                         for sub, (ss, keep, cath, n_full) in sorted(resp.items())])
print(cell_tab.to_string(index=False))
print("\nResponding cells:", int(cell_tab.responding.sum()), "; by dye:", cell_tab.groupby("dye").responding.sum().to_dict())
geo = D.groupby(["dye", "subject", "cell"])[["R_area_um", "R_perim_um", "dist_centre_px"]].mean()
print(f"Cell radius (um, pixel {pef.PIXEL_UM:.4f} um): equal-area {geo.R_area_um.min():.2f} to {geo.R_area_um.max():.2f}, "
      f"mean {geo.R_area_um.mean():.2f}; perimeter/2pi {geo.R_perim_um.min():.2f} to {geo.R_perim_um.max():.2f}, mean {geo.R_perim_um.mean():.2f}")
print("Mean radius by dye (equal-area):", geo.groupby("dye").R_area_um.mean().round(2).to_dict())

Every field of view holds several cells (3 to 6 found), while the paper analysed one cell per
experiment and does not say which. The cell analysed in each experiment cannot be identified
from the files. Three readings are run: (i) the mean of the experiment's responding cells, with
the experiment as the unit (n = 5 per dye); (ii) the responding cell nearest the image centre,
one per experiment; (iii) every responding cell as a unit. The field direction, which the files
do not record, is found from the data: in every experiment the side whose emission rises (the
cathode-facing side, Methods 2.3) lies between 94 and 106 degrees of the image's +x axis, that
is towards the bottom of the image. With a 16-um camera pixel and the 60x objective, the
mean radius is about 6.7 um. The paper gives about 8 um for CHO cells as a typical value,
so the pixel size is consistent with the data but not shown by them.

In [ ]:
# Recordings deposited twice: the frame-to-frame fluctuation of the laser makes each set's
# sequence of whole-image means a fingerprint. Compare every pair of sets.
x = fm.copy()
x[~good] = np.nan
x = (x - np.nanmean(x, 1, keepdims=True)) / np.nanstd(x, 1, keepdims=True)
x = np.nan_to_num(x)
cc = x @ x.T / x.shape[1]
np.fill_diagonal(cc, np.nan)
iu = np.triu_indices(len(sets), 1)
print(f"Correlation of frame-mean sequences over {len(iu[0])} pairs of sets: max {np.nanmax(cc[iu]):.2f}, "
      f"99th percentile {np.nanpercentile(cc[iu], 99):.2f}")
identical = [(i, j) for i, j in zip(*iu) if np.array_equal(sets[i]["frame_mean"], sets[j]["frame_mean"])]
print("Pairs with identical frame means:", len(identical))
prof0 = {}
for s in sets:
    for c in s["cells"]:
        prof0.setdefault(round(float(c["prof"][1].sum()), 3), []).append(s["asset_id"])
print("Cell profiles shared by two files:", sum(len(v) > 1 for v in prof0.values()))

No two sets share their frame-by-frame laser fluctuation or a cell profile, so no recording is
deposited twice. The cells of each experiment are linked from set to set by position
(within 10 pixels); the same object is found in all 19 sets, so a cell is treated as the same
across the sets of an experiment.

In [ ]:
# Shared clock: the pulse starts at image 41 (index 40) in every set (Methods 2.3: 40 baseline
# images, then 60 with the pulse). Polarization (cathodic minus anodic dF, halved) in 10-image
# windows either side of a boundary; the step should be largest at the true onset, and absent at
# the same boundary in the lowest-voltage sets if the response is a response.
def polarization_frames(df):
    return np.stack([(a - b) / 2 for a, b in zip(df.frames_cath, df.frames_anod)])

onsets, steps = [], []
for (sub, volt), g in D.groupby(["subject", "volt"]):
    pol = np.nanmean(polarization_frames(g), 0)
    st = {b: np.nanmean(pol[b:b + 10]) - np.nanmean(pol[b - 10:b]) for b in range(15, 86)}
    onsets.append(dict(subject=sub, volt=volt, onset=max(st, key=lambda b: abs(st[b])), step40=st[40],
                       step30=st[30], step50=st[50]))
on = pd.DataFrame(onsets)
print("Boundary with the largest polarization step, sets at 50 V and above:")
print(on[on.volt >= 50].onset.value_counts().sort_index().to_string())
print("\nMean step (% dF) at boundaries 30, 40, 50, by voltage group:")
print(on.assign(group=pd.cut(on.volt, [0, 25, 60, 120, 250])).groupby("group", observed=True)[["step30", "step40", "step50"]].mean().round(2))

In every set at 50 V and above, the polarization step is largest at index 40 or within a frame
or two of it, and it grows with voltage; at boundaries 30 and 50 the step is near zero. The
images are therefore on the clock the paper describes: 40 images without the pulse, then 60
with it. Because there is one stream per file, no other stream needs aligning.

In [ ]:
# Field strength per volt implied by the low-field response under the paper's calibration
# (Fig. 3: 26 and 9.1% dF per 100 mV) and Eq. (1). Per cell, the slope of the half-difference
# between the poles (which cancels the uniform bleaching offset) over the sets at 25 to 60 V.
rows = []
for (dye, sub, cell), g in D.groupby(["dye", "subject", "cell"]):
    g = g[g.volt <= 60]
    half = (g.dF_cath - g.dF_anod) / 2
    slope = np.sum(half * g.volt) / np.sum(g.volt ** 2)
    R = g.R_area_um.mean()
    rows.append(dict(dye=dye, subject=sub, cell=cell, k=slope / (pef.SENS[dye] / 100 * pef.impc_mV(1.0, R, 0))))
kk = pd.DataFrame(rows)
print("Implied V/cm per volt, per cell, if the paper's dye sensitivities hold:")
print(kk.groupby("dye").k.describe().round(2))
print("\nField at the midpoint between two 0.5-mm rods (two-dimensional, helper rod_field_per_volt), V/cm per volt:")
for sp in (1.6, 1.0):
    print(f"  spacing {sp} mm read as centre to centre: {pef.rod_field_per_volt(sp):.2f}; as the gap between the rods: "
          f"{pef.rod_field_per_volt(sp, gap=True):.2f}")
print("\nReadings: top", round(pef.FIELD_PER_VOLT["top"], 2), "(adopted)  bottom", round(pef.FIELD_PER_VOLT["bottom"], 2))
low = D[D.volt == 25]
print(f"\nAt 25 V, mean of the two poles (bleaching offset): Di-8-ANEPPS {((low.dF_cath + low.dF_anod) / 2)[low.dye == 'Di8'].mean():.1f}%, "
      f"FluoVolt {((low.dF_cath + low.dF_anod) / 2)[low.dye == 'FV'].mean():.1f}%")

Two checks bear on the factor, and they point in different directions.

- If the paper's dye sensitivities hold, the low-field response implies a median of 1.5 to 1.6
  V/cm per volt in each dye, near `bottom` (1.72). This uses the sensitivities that
  fig3-dye-sensitivity tests, so a factor chosen this way cannot be used to test them.
- The electrode geometry of Methods 2.2 (rods 0.5 mm in diameter, 1.6 mm apart) gives 4.64 to
  7.19 V/cm per volt at the midpoint in a two-dimensional model, depending on whether 1.6 mm is
  the gap or the centre distance. The files' 1.00 mm gives 8.02 to 17.54. `top` (5.6) lies in
  the range for the paper's geometry; `bottom` is 2.7 to 10 times below every geometric value.

The `top` reading is adopted as an assumption. It is the one the geometry supports, it does
not use any quantity a claim tests, and it puts the dataset's highest amplitude on the paper's
highest field. The two-dimensional model ignores the dish bottom and the rod ends, so it gives
the order of the factor, not its value. `bottom` is shown as a variant, and every criterion is
run over factors from 1.5 to 8 in the section before the summary. Under `top` the sets cover
140 to 1400 V/cm, so the paper's 43 V/cm level has no set. At 25 V the mean of the two poles is
-3.9% (Di-8-ANEPPS) and -2.1% (FluoVolt). This uniform offset matches the bleaching the paper
reports in its own Di-8-ANEPPS dF at 43 V/cm (Results 3.2, paragraph 3: about 5% added at the
anodic pole and removed at the cathodic pole).

In [ ]:
# Ranges of the variables used.
for col in ["dF_cath", "dF_anod", "pi_cath", "pi_anod", "f10_cath", "f10_anod"]:
    print(f"{col:9s} by dye: " + ", ".join(f"{d} {g[col].min():.1f} to {g[col].max():.1f} %" for d, g in D.groupby("dye")))
print("Without background subtraction:", ", ".join(f"{d} cathodic {g.dF_cath.min():.1f} to {g.dF_cath.max():.1f} %" for d, g in D_raw.groupby("dye")))
print(f"Camera offset only ({OFFSET:.0f} counts):", ", ".join(f"{d} cathodic {g.dF_cath.min():.1f} to {g.dF_cath.max():.1f} %" for d, g in D_off.groupby("dye")))

In [ ]:
# Pole dF against voltage, mean of experiment means (reading i), both dyes.
def units(df, reading):
    # (i) experiment mean of its cells; (ii) the cell nearest the image centre; (iii) every cell.
    if reading == "experiment":
        g = df.groupby(["dye", "subject", "volt"], as_index=False)
        out = g.mean(numeric_only=True)
        out["unit"] = out.subject
        return out
    if reading == "centre":
        pick = df.groupby("subject").apply(lambda g: g.loc[g.dist_centre_px.idxmin(), "cell"])
        out = df[df.cell == df.subject.map(pick)].copy()
        out["unit"] = out.subject
        return out
    out = df.copy()
    out["unit"] = out.subject + "/" + out.cell.astype(str)
    return out

READINGS = ["experiment", "centre", "cells"]
K = dict(pef.FIELD_PER_VOLT)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, dye in zip(axes, ["FV", "Di8"]):
    u = units(D[D.dye == dye], "experiment")
    for pole, col in [("cathodic", "dF_cath"), ("anodic", "dF_anod")]:
        m = u.groupby("volt")[col].agg(["mean", "sem"])
        ax.errorbar(m.index, m["mean"], m["sem"], marker="o", ms=3, label=pole)
    ax.set_title({"FV": "FluoVolt", "Di8": "Di-8-ANEPPS"}[dye] + " (n = 5 experiments)")
    ax.set_xlabel("pulse amplitude (V)")
    sec = ax.secondary_xaxis("top", functions=(lambda v: v * K["top"], lambda e: e / K["top"]))
    sec.set_xlabel("V/cm, top reading (adopted)")
    ax.axhline(0, color="0.7", lw=0.5)
axes[0].set_ylabel("collapsed pole dF (%)")
axes[0].legend()
plt.tight_layout()
plt.show()
mm = units(D, "experiment").groupby(["dye", "volt"])[["dF_cath", "dF_anod"]].agg(["mean", "sem"])
print("Plotted means, range:", mm.xs("mean", axis=1, level=1).min().round(1).to_dict(),
      mm.xs("mean", axis=1, level=1).max().round(1).to_dict(), "; no point outside the axes")
tab = mm.unstack("dye").round(1)
tab.index = [f"{v} V = {v * K['top']:.0f} V/cm top, {v * K['bottom']:.0f} bottom" for v in tab.index]
print("\nCollapsed pole dF (%) at every set, mean and s.e. of experiment means (n = 5), background subtracted:")
print(tab.to_string())
me = units(D_edge, "experiment").groupby(["dye", "volt"])[["dF_cath", "dF_anod"]].mean().unstack("dye").round(1)
print("\nThe same means with the edge trace (variant):")
print(me.to_string())
geo_e = D_edge.groupby(["dye", "subject", "cell"]).R_area_um.mean()
print(f"\nEqual-area radius of the edge trace: mean {geo_e.mean():.2f} um ({geo_e.min():.2f} to {geo_e.max():.2f}); peak trace {D.groupby(['dye', 'subject', 'cell']).R_area_um.mean().mean():.2f} um")

### Check of the automated membrane trace

The pole dF magnitudes below are about half the paper's at most levels, so the trace is checked
on the 120 V and 250 V sets of every experiment (re-read from the archive): the
background-subtracted pole dF on 1-pixel rings from 8 pixels inside to 8 outside the trace.
The 6-pixel trace spans offsets -2.5 to +2.5. If the trace sat off the membrane, the pole dF
would rise when the ring moves onto it.

In [ ]:
def radial_one(job):
    sub, volt = job
    ss, keep, cath, _ = resp[sub]
    j = [s["volt"] for s in ss].index(volt)
    return sub, volt, ss[j], [row[j] for row in keep], pef.radial_poles(ss[j], [row[j] for row in keep], cath)

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(10) as ex:
    radial = list(ex.map(radial_one, [(sub, v) for sub in sorted(resp) for v in (120, 250)]))
rows = []
for sub, volt, s, cis, r in radial:
    off = r["offsets"]
    for k, c in enumerate(cis):
        tr = D[(D.subject == sub) & (D.volt == volt) & (D.cell == k)].iloc[0]
        for pole in ("cath", "anod"):
            v = r["cells"][k][pole]
            sign = 1 if pole == "cath" else -1
            band = np.abs(off) <= 3
            rows.append(dict(dye=s["dye"], subject=sub, volt=volt, cell=k, pole=pole, trace=tr[f"dF_{pole}"],
                             ring0=v[off == 0][0], best_in_band=sign * np.max(sign * v[band]),
                             best_offset_in_band=int(off[band][np.argmax(sign * v[band])]),
                             best_8px=sign * np.max(sign * v), best_offset_8px=int(off[np.argmax(sign * v)])))
rc = pd.DataFrame(rows)
print("Pole dF (%): the trace, the ring on the trace, and the largest |dF| over rings within 3 and 8 pixels;"
      " mean over experiments of the experiment means:")
print(rc.groupby(["dye", "volt", "subject", "pole"])[["trace", "ring0", "best_in_band", "best_8px"]].mean()
      .groupby(["dye", "volt", "pole"]).mean().round(1).to_string())
print("\nLargest single-cell values, rings within 3 pixels:")
print(rc.groupby(["dye", "pole"]).best_in_band.agg(["min", "max"]).round(1).to_string())
print("\nOffset (pixels) of the largest |dF| within 8 pixels, counts:", rc.best_offset_8px.value_counts().sort_index().to_dict())
print("Dark-frame level of these sets (camera offset):", f"{np.nanmin([x[4]['dark'] for x in radial]):.0f} to {np.nanmax([x[4]['dark'] for x in radial]):.0f} counts")

fig, axes = plt.subplots(2, 5, figsize=(15, 6.4))
for ax, (sub, volt, s, cis, r) in zip(axes.ravel(), [x for x in radial if x[1] == 250]):
    ax.imshow(r["base"], cmap="gray", vmin=np.percentile(r["base"], 1), vmax=np.percentile(r["base"], 99.8))
    th = np.deg2rad(np.arange(361))
    for ci in cis:
        c = s["cells"][ci]
        for key, col in [("rmem", "r-"), ("rmem_edge", "c-")]:
            rm = np.r_[c[key], c[key][0]]
            ax.plot(c["cx"] + np.cos(th) * rm, c["cy"] + np.sin(th) * rm, col, lw=0.6)
    ax.set_title(f"{sub[-3:]} {s['dye']} 250 V, baseline", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Baseline images shown with the 1st to 99.8th percentile as the grey range; the lines are the trace centres of the analysed cells, red the peak trace (main), cyan the edge trace (variant).")

On the ring at the trace, experiment-mean pole dF is within 3 points of the trace value. The
largest |dF| over rings within 3 pixels of the trace is 5.5 to 7.7 points larger (it picks the
largest of seven noisy values, so it is biased upwards). For FluoVolt it reaches 43.4% and
-57.1% at the most, against the paper's 66% and -83%. For Di-8-ANEPPS it reaches 30.9% and
-35.0%, against 57% and -55%. Rings up to 8 pixels outside give larger values, because there
F0 approaches the background. They stay below the paper's FluoVolt values (50.9%, -65.0%) and
its Di-8-ANEPPS cathodic value (36.7%); the Di-8-ANEPPS anodic value reaches -55.3% at 250 V. Single cells reach 79.5% and -105.5% (FluoVolt) and
53.7% and -84.4% (Di-8-ANEPPS) within 3 pixels, so the paper's values lie inside the spread of
single cells. A misplaced trace does not explain the shortfall of the means.

The overlay shows two faults of the peak trace (red). In some cells it is drawn inwards, where
bright spots inside the cell outshine the rim. In c7 two touching cells are found as one
object. A second trace that follows the cell edge (cyan; helper `trace_edge`) is run as a
variant. It follows most outlines, but bridges into a touching neighbour in a few cells. With it the largest experiment-mean pole values over the 19 sets are 26.9% and -41.3%
(FluoVolt) and 25.1% and -30.4% (Di-8-ANEPPS), no nearer the paper's. Its mean radius is 9.90
um, against 6.72 um for the peak trace. The sweep before the summary runs every criterion with
both traces.

## fig3-dye-sensitivity

> Fitted linearly through the origin over -150 to +150 mV IMPC, FluoVolt changes by 26% and
> Di-8-ANEPPS by 9.1% dF per 100 mV, with correlation coefficients 0.96 and 0.78.

Every perimeter point of every analysed cell, in the sets from 43 to 360 V/cm, against IMPC from
Eq. (1) with the cell's own radius and tau = 44.5 ns/um x R (reference [20]). Least squares
through the origin over IMPC from -150 to +150 mV. Readings run: field mapping (top, adopted;
bottom), background (b subtracted, a none), radius (equal-area, perimeter / 2 pi), cells (all
responding cells, or the cell nearest the image centre), and sets (every set up to 360 V/cm, or
only the sets nearest the paper's levels 43, 150, 230 and 360 V/cm). The correlation
coefficient is read as (a) Pearson r and (b) the square root of the uncentred R2.

In [ ]:
def calib_points(dye, mapping, background, radius, cells, which_sets, detrend=False):
    df = (D if background else D_raw)
    df = df[df.dye == dye]
    if cells == "centre":
        df = units(df, "centre")
    k = K[mapping]
    if which_sets == "levels":
        lv = pef.nearest_sets(df.volt, k)
        df = df[df.volt.isin([v for L, v in lv.items() if L <= 360])]
    else:
        df = df[df.volt * k <= 360]
    th = np.arange(360)
    X = np.concatenate([pef.impc_mV(v * k, r, th) for v, r in zip(df.volt, df[f"R_{radius}_um"])])
    # detrend: remove each profile's mean, the uniform (bleaching) offset; a variant.
    Y = np.concatenate([p - p.mean() if detrend else p for p in df.profile.values])
    return X, Y, df

def fit0(X, Y, lim=150):
    m = np.abs(X) <= lim
    x, y = X[m], Y[m]
    if len(x) == 0:
        return dict(slope=np.nan, r_pearson=np.nan, r_uncentred=np.nan, n_points=0)
    return dict(slope=100 * np.sum(x * y) / np.sum(x * x), r_pearson=np.corrcoef(x, y)[0, 1],
                r_uncentred=np.sum(x * y) / np.sqrt(np.sum(x * x) * np.sum(y * y)), n_points=int(m.sum()))

EXP3 = {"FV": (26, 2.6, 0.96, 0.096), "Di8": (9.1, 0.91, 0.78, 0.078)}

def fig3_row(mapping, background=True, radius="area", cells="all", which="all"):
    out = dict(mapping=mapping, bg="b" if background else "a", radius=radius, cells=cells, sets=which)
    ok = True
    for dye in ["FV", "Di8"]:
        X, Y, df = calib_points(dye, mapping, background, radius, cells, which)
        f = fit0(X, Y)
        s0, ts, r0, tr = EXP3[dye]
        out.update({f"{dye}_slope": f["slope"], f"{dye}_r_pearson": f["r_pearson"], f"{dye}_r_uncentred": f["r_uncentred"],
                    f"{dye}_n_sets": df.volt.nunique(), f"{dye}_n_cells": df.groupby(["subject", "cell"]).ngroups,
                    f"{dye}_n_points": f["n_points"]})
        ok &= bool(abs(f["slope"] - s0) <= ts and (abs(f["r_pearson"] - r0) <= tr or abs(f["r_uncentred"] - r0) <= tr))
    out["reproduced"] = ok if out["FV_n_points"] and out["Di8_n_points"] else None
    return out

cal = pd.DataFrame([fig3_row(mapping, background, radius, cells, which)
                    for mapping in ["top", "bottom"] for background in [True, False] for radius in ["area", "perim"]
                    for cells in ["all", "centre"] for which in ["all", "levels"]])
print(cal.round(3).to_string(index=False))

In [ ]:
# A reading reproduces when, for both dyes, the slope is within tolerance and r is within
# tolerance under (a) or (b).
print("Readings where both dyes meet the criterion:", int((cal.reproduced == True).sum()), "of", len(cal))
for mapping in ["top", "bottom"]:
    s = cal[cal.mapping == mapping]
    print(f"{mapping}: FluoVolt slope {s.FV_slope.min():.2f} to {s.FV_slope.max():.2f}, Di-8-ANEPPS {s.Di8_slope.min():.2f} to {s.Di8_slope.max():.2f};"
          f" r (Pearson) FluoVolt {s.FV_r_pearson.min():.3f} to {s.FV_r_pearson.max():.3f}, Di-8-ANEPPS {s.Di8_r_pearson.min():.3f} to {s.Di8_r_pearson.max():.3f}")
main = cal[(cal.mapping == "top") & (cal.bg == "b") & (cal.radius == "area") & (cal.cells == "all") & (cal.sets == "all")]
print("\nAdopted reading (top, background subtracted, equal-area radius, all cells, all sets):")
print(main.round(3).T.to_string())
by_exp = []
for dye in ["FV", "Di8"]:
    X, Y, df = calib_points(dye, "top", True, "area", "all", "all")
    sub = np.repeat(df.subject.values, 360)
    for s in np.unique(sub):
        by_exp.append(dict(dye=dye, subject=s, **fit0(X[sub == s], Y[sub == s])))
print("\nBy experiment, adopted reading:")
print(pd.DataFrame(by_exp).round(3).to_string(index=False))
# Adopted reading on a fine grid of factors around those that give the paper's slopes. The slope
# does not scale as 1 / factor, because the sets at or below 360 V/cm change with the factor.
fine = []
for k in np.round(np.arange(1.40, 2.301, 0.02), 2):
    K[f"fine{k:g}"] = k
    r = fig3_row(f"fine{k:g}")
    fine.append(dict(factor=k, sets_FV=r["FV_n_sets"], FV_slope=r["FV_slope"], FV_r=r["FV_r_pearson"],
                     FV_r_unc=r["FV_r_uncentred"], Di8_slope=r["Di8_slope"], Di8_r=r["Di8_r_pearson"],
                     Di8_r_unc=r["Di8_r_uncentred"], reproduced=r["reproduced"]))
fine = pd.DataFrame(fine)
print("\nAdopted reading over factors 1.40 to 2.30 V/cm per volt:")
print(fine.round(3).to_string(index=False))
for dye in ["FV", "Di8"]:
    s0, ts = EXP3[dye][:2]
    ok = fine.factor[(fine[f"{dye}_slope"] - s0).abs() <= ts]
    print(f"{dye}: slope within {s0} +- {ts} at factors {list(ok)}")
print("Factors on this grid that meet the criterion for both dyes:", list(fine.factor[fine.reproduced == True]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, dye in zip(axes, ["FV", "Di8"]):
    X, Y, _ = calib_points(dye, "top", True, "area", "all", "all")
    ax.plot(X[::7], Y[::7], ",", alpha=0.3)
    f = fit0(X, Y)
    g = np.linspace(X.min(), X.max(), 10)
    ax.plot(g, f["slope"] / 100 * g, "k--", label=f"fit {f['slope']:.1f}%/100 mV")
    ax.plot(g, EXP3[dye][0] / 100 * g, "r:", label=f"paper {EXP3[dye][0]}%/100 mV")
    ax.set_ylim(-100, 100)
    ax.set_xlabel("IMPC (mV), top mapping")
    ax.set_title({"FV": "FluoVolt", "Di8": "Di-8-ANEPPS"}[dye])
    ax.legend(fontsize=8)
    print(dye, f"points {len(X)}, every 7th plotted; dF range {Y.min():.0f} to {Y.max():.0f} %;"
          f" outside the axes {(np.abs(Y) > 100).sum()} points; IMPC range {X.min():.0f} to {X.max():.0f} mV")
axes[0].set_ylabel("dF (%)")
plt.tight_layout()
plt.show()

**Result: not reproduced (under the assumed factor).** Under the adopted reading (top mapping,
background subtracted, equal-area radius, all responding cells, the four sets from 140 to 336
V/cm), FluoVolt gives 6.10% dF per 100 mV (expected 26 +- 2.6) with r = 0.775 (Pearson) and
0.743 (uncentred), against 0.96 +- 0.096. Di-8-ANEPPS gives 2.37% (expected 9.1 +- 0.91) with r
= 0.481 and 0.402, against 0.78 +- 0.078. Over the 16 top-mapping readings the slopes are 2.72 to
6.10 and 0.83 to 2.37, and by experiment 5.36 to 6.64 and 2.20 to 2.53. The slope falls as the
factor rises, but not as one over the factor, because the sets at or below 360 V/cm change with
it. On the grid from 1.40 to 2.30 V/cm per volt, the FluoVolt slope is within tolerance at 1.40
to 1.76 and the Di-8-ANEPPS slope at 1.82 to 2.16, and r is within tolerance for both dyes at
every factor (FluoVolt 0.90 to 0.92, Di-8-ANEPPS 0.79 to 0.84). No factor on that grid or in the
sweep from 1.5 to 8 meets the criterion for both dyes, so the result is not reproduced at the
factors run. The factors that match each of the paper's slopes, about 1.4 to 2.2, are near the
1.5 to 1.6 that the low-field response implies under the paper's calibration (data check) and
below the 4.64 to 7.19 of the electrode geometry. Under the `bottom` variant, which was set from
this response, the slopes are 23.98 and 10.52 and r is 0.908 and 0.824. None of the 32 readings
meets the criterion for both dyes. At the geometric factors (4.64 to 7.19) the slopes are 4.4
to 8.0 and 1.6 to 3.2. With the edge trace (data check) they are 4.5 and 1.8 at the adopted
factor.

## fig3-fluovolt-nonlinear

> FluoVolt departed from linearity from about +-200 mV IMPC, while Di-8-ANEPPS stayed nearly
> linear from -500 to +500 mV; so FluoVolt's early departure reflects its dynamic range, not
> electroporation.

The points and line of fig3-dye-sensitivity (adopted reading; the bottom mapping as a variant).
Smooths: LOWESS with span (a) 0.3, (b) 0.5, (c) 0.75, and (d) means in 50-mV bins. A departure is
the smooth lying closer to zero than the line by more than 10% of the line's value. Checked on a
10-mV grid; LOWESS is evaluated with `delta` set to 1% of the IMPC range, which interpolates
between nearby fitted points (a standard speed-up that cannot change the outcome).

The fit is through the origin, so a uniform offset of the profile (the bleaching seen at 25 V,
data check) moves the smooth off the line near zero. The paper reports that bleaching in its
own Di-8-ANEPPS dF (Results 3.2, paragraph 3) and does not say it removed it, so the main reading
keeps it. As a variant, each profile's mean over the 360 points (the uniform offset; the cosine
averages to zero) is removed before the fit and the smooth (`detrend`).

In [ ]:
def smooth(X, Y, how):
    o = np.argsort(X)
    X, Y = X[o], Y[o]
    if how == "bins":
        edges = np.arange(np.floor(X.min() / 50) * 50, X.max() + 50, 50)
        idx = np.digitize(X, edges)
        xs = np.array([X[idx == i].mean() for i in np.unique(idx) if (idx == i).sum() >= 20])
        ys = np.array([Y[idx == i].mean() for i in np.unique(idx) if (idx == i).sum() >= 20])
        return xs, ys
    fr = lowess(Y, X, frac=how, it=0, delta=0.01 * (X.max() - X.min()), return_sorted=True)
    return fr[:, 0], fr[:, 1]

def departure(X, Y, slope, how):
    xs, ys = smooth(X, Y, how)
    grid = np.arange(np.ceil(xs.min() / 10) * 10, xs.max(), 10)
    sm = np.interp(grid, xs, ys)
    line = slope / 100 * grid
    dep = (np.abs(line) - np.sign(line) * sm) > 0.1 * np.abs(line)
    dep[grid == 0] = False
    return grid, sm, line, dep

def nonlinear_rows(mapping, detrend=False, smooths=(0.3, 0.5, 0.75, "bins")):
    res, cur = [], {}
    for dye in ["FV", "Di8"]:
        X, Y, _ = calib_points(dye, mapping, True, "area", "all", "all", detrend=detrend)
        slope = fit0(X, Y)["slope"]
        for how in smooths:
            grid, sm, line, dep = departure(X, Y, slope, how)
            cur[(mapping, detrend, dye, how)] = (grid, sm, line, dep)
            r = dict(mapping=mapping, detrend=detrend, dye=dye, smooth=how, slope=slope, data_range=f"{grid.min():.0f} to {grid.max():.0f}")
            if dye == "FV":
                hi = np.abs(grid) >= 220
                both = (grid >= 220).any() and (grid <= -220).any()
                r.update(grid_pts_220_to_end=int(hi.sum()), ok=bool(both and dep[hi].all()), departing_pts=int(dep[hi].sum()))
            else:
                inr = np.abs(grid) <= 500
                r.update(grid_pts_in_range=int(inr.sum()), ok=not bool(dep[inr].any()), departing_pts=int(dep[inr].sum()),
                         departing_at=", ".join(f"{g:.0f}" for g in grid[inr & dep][:12]))
            res.append(r)
    return pd.DataFrame(res), cur

def nonlinear_verdict(t):
    # Reproduced when every smooth gives FluoVolt departing and Di-8-ANEPPS not departing.
    return bool(t.ok.all())

t0 = time.time()
nl, curves = [], {}
for mapping in ["top", "bottom"]:
    for detrend in [False, True]:
        t, c = nonlinear_rows(mapping, detrend)
        nl.append(t)
        curves.update(c)
nl = pd.concat(nl)
print(nl.round(2).to_string(index=False))
print("\nReproduced (every smooth), by mapping and offset removal:")
print(nl.groupby(["mapping", "detrend"]).apply(nonlinear_verdict).to_string())
print(f"({time.time() - t0:.0f} s)")

In [ ]:
# Size of the departure: smooth as % of the line at chosen IMPC values, adopted mapping.
rows = []
for detrend in [False, True]:
    for dye in ["FV", "Di8"]:
        for how in [0.3, 0.5, 0.75, "bins"]:
            grid, sm, line, dep = curves[("top", detrend, dye, how)]
            r = {"detrend": detrend, "dye": dye, "smooth": how}
            for g in [-300, -220, -150, -100, 100, 150, 220, 300]:
                if grid.min() <= g <= grid.max():
                    r[g] = round(100 * np.interp(g, grid, sm) / np.interp(g, grid, line), 1)
            rows.append(r)
print("Smooth as % of the fitted line (100 = on the line; below 90 = departure), top mapping:")
print(pd.DataFrame(rows).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, dye in zip(axes, ["FV", "Di8"]):
    for how in [0.3, 0.5, 0.75, "bins"]:
        grid, sm, line, dep = curves[("top", False, dye, how)]
        ax.plot(grid, sm, label=str(how))
    ax.plot(grid, line, "k--", label="line")
    ax.set_title({"FV": "FluoVolt", "Di8": "Di-8-ANEPPS"}[dye] + ", top mapping")
    ax.set_xlabel("IMPC (mV)")
    ax.legend(fontsize=7)
    print(dye, "smoothed dF range", f"{min(curves[('top', False, dye, h)][1].min() for h in [0.3, 0.5, 0.75, 'bins']):.1f} to "
          f"{max(curves[('top', False, dye, h)][1].max() for h in [0.3, 0.5, 0.75, 'bins']):.1f} %; all inside the axes")
axes[0].set_ylabel("smoothed dF (%)")
plt.tight_layout()
plt.show()

**Result: not reproduced (under the assumed factor).** Under the top mapping the FluoVolt smooth
does not depart from the line at |IMPC| of 220 mV and above under any LOWESS span (0 departing
grid points of 48), and in 12 of 42 points with 50-mV bins. It lies above the line: 107 to 113%
of the line at +220 mV and 128 to 134% at -220 mV. Di-8-ANEPPS departs at 22 to 23 grid points
under every smooth, on the positive side near zero, where its smooth is -27 to 0.5% of the line
at +100 mV. With each profile's offset removed, FluoVolt still does not depart, and Di-8-ANEPPS
departs under span 0.3 and bins (2 and 6 points) but not under 0.5 and 0.75. So no reading meets
the criterion. Under the bottom variant FluoVolt departs at every point from 220 mV under every
smooth, but Di-8-ANEPPS departs at 58 to 64 points (40 to 50 with the offset removed), so the
criterion fails there as well. No factor in the sweep meets it.

## fig2a-fluovolt-saturation

> With FluoVolt, pole dF rose with field strength until it saturated at 66% (255 mV) at 360
> V/cm at the cathodic pole and -83% (-320 mV) at 570 V/cm at the anodic pole; stronger fields
> did not increase the anodic response and decreased the cathodic one.

Per-image pole dF (the Fig. 2 measure, Methods 2.3), averaged over the 60 pulse images of each
set, per cell, then across units. The paper's levels are the sets nearest them (helper,
`nearest_sets`, within 15%). Both mappings, three cell readings, and three background readings
are run: (b) the cell-free median (adopted), (a) none, and the camera offset alone (the
dark-frame level), a variant.

In [ ]:
def verdict(*conds):
    # Combine conditions: False if any fails, None (untestable) if any is missing, else True.
    if any(c is False for c in conds):
        return False
    if any(c is None for c in conds):
        return None
    return True

def level_table(df, mapping, cols, reading):
    lv = pef.nearest_sets(df.volt, K[mapping])
    u = units(df, reading)
    out = {}
    for L, v in lv.items():
        g = u[u.volt == v]
        out[L] = {c: (g[c].mean(), g[c].sem(), len(g)) for c in cols}
    return out, lv

def show_levels(dye, cols, background=True, readings=READINGS, mappings=("top", "bottom")):
    df = {True: D, False: D_raw, "offset": D_off}[background]
    df = df[df.dye == dye]
    rows = []
    for mapping in mappings:
        for reading in readings:
            tab, lv = level_table(df, mapping, cols, reading)
            for L, d in tab.items():
                r = dict(mapping=mapping, cells=reading, level=L, volt=lv[L], n=d[cols[0]][2])
                for c in cols:
                    r[c] = round(d[c][0], 1)
                    r[c + "_se"] = round(d[c][1], 1)
                rows.append(r)
    return pd.DataFrame(rows)

fv2 = show_levels("FV", ["pi_cath", "pi_anod"])
print(fv2.to_string(index=False))

In [ ]:
def check_fig2a(t):
    t = t.set_index("level")
    c360 = t.pi_cath.get(360, np.nan)
    a570 = t.pi_anod.get(570, np.nan)
    hi = [L for L in (1000, 1200, 1400) if L in t.index]
    out = dict(cath_360=c360, cath_360_ok=bool(abs(c360 - 66) <= 6.6) if not np.isnan(c360) else None, anod_570=a570,
               anod_570_ok=bool(abs(a570 - (-83)) <= 8.3) if not np.isnan(a570) else None,
               cath_360_is_max=bool(c360 >= t.pi_cath.max()) if not np.isnan(c360) else None,
               max_cath=t.pi_cath.max(), max_cath_level=int(t.pi_cath.idxmax()),
               min_anod=t.pi_anod.min(), min_anod_level=int(t.pi_anod.idxmin()), levels_1000_1400=len(hi))
    out["anod_not_more_neg"] = bool(all(t.pi_anod[L] >= a570 - 8.3 for L in (780, 1000, 1200, 1400) if L in t.index)) if hi and not np.isnan(a570) else None
    out["cath_hi_below_360"] = bool(all(t.pi_cath[L] < c360 for L in hi)) if hi and not np.isnan(c360) else None
    out["reproduced"] = verdict(out["cath_360_ok"], out["anod_570_ok"], out["cath_360_is_max"], out["anod_not_more_neg"], out["cath_hi_below_360"])
    return out

def fig2a_sat(mapping, reading="experiment", background=True):
    t = show_levels("FV", ["pi_cath", "pi_anod"], background, readings=[reading], mappings=[mapping])
    return check_fig2a(t)

rows = [dict(bg=bgname, mapping=mapping, cells=reading, **fig2a_sat(mapping, reading, background))
        for bgname, background in [("b", True), ("a", False), ("offset", "offset")]
        for mapping in ["top", "bottom"] for reading in READINGS]
print(pd.DataFrame(rows).round(1).to_string(index=False))
pe = D[D.dye == "FV"].groupby(["subject", "volt"])[["pi_cath", "pi_anod"]].mean().unstack("subject")
lv = pef.nearest_sets(D.volt, K["top"])
print("Per experiment, top mapping levels (mean of each experiment's cells):")
print(pe.loc[list(lv.values())].round(1).to_string())
print(f"\nLargest per-experiment cathodic value at any set: {pe.pi_cath.max().max():.1f}%; most negative anodic: {pe.pi_anod.min().min():.1f}%")

**Result: not reproduced.** At 360 V/cm (60 V under the top mapping) the per-image cathodic dF
is 23.5% (experiment means; 20.1 for the cell nearest the centre, 24.1 over all cells), against
66 +- 6.6. At 570 V/cm the anodic dF is -35.8% (-33.7 and -38.6), against -83 +- 8.3. The cathodic
maximum is 35.9% at 570 V/cm, not at 360. Over all 19 sets the largest experiment-mean cathodic
value is 36.1% and the most negative anodic value -49.5%. Per experiment the values reach 43.6% and
-61.1% at most. With the camera offset as background they are 17.3% and -25.2%, and without
background 12.1% and -17.9%. The magnitudes fail under every mapping, cell and background
reading and every factor in the sweep. Under the top mapping the cathodic response falls from
35.9% at 570 V/cm to 21.2% at 1400 V/cm and the anodic one grows from -35.8% to -49.4%.

## fig2a-fluovolt-stable

> FluoVolt measurements were stable without pulses (no significant bleaching or
> photosensitization), and the response within each set stayed stable through the 60 pulses,
> even at 1000 V/cm and above.

Per set and pole, the start and end of the 40 baseline images, and of the 60 pulse images,
compared by a 2-tailed paired t-test over units at p < 0.05. Start and end are (a) the first
and the last lit image, (b) the first 10 and the last 10. The sets are those nearest the
paper's levels (main), with all 19 sets as a variant.

In [ ]:
def start_end(frames, lo, hi, mode):
    seg = frames[lo:hi]
    ok = np.nonzero(~np.isnan(seg))[0]
    if mode == "a":
        return seg[ok[0]], seg[ok[-1]]
    return np.nanmean(seg[:10]), np.nanmean(seg[-10:])

def stability(dye, phase, mapping, reading, mode, sets_sel="levels", background=True):
    df = (D if background else D_raw)
    df = df[df.dye == dye].copy()
    lo, hi = (0, pef.N_BASE) if phase == "baseline" else (pef.N_BASE, 100)
    for pole in ["cath", "anod"]:
        se = np.array([start_end(f, lo, hi, mode) for f in df[f"frames_{pole}"]])
        df[f"s_{pole}"], df[f"e_{pole}"] = se[:, 0], se[:, 1]
    u = units(df, reading)
    if sets_sel == "levels":
        lv = pef.nearest_sets(df.volt, K[mapping])
        pick = {v: L for L, v in lv.items()}
    else:
        pick = {v: round(v * K[mapping]) for v in sorted(df.volt.unique())}
    rows = []
    for v, L in pick.items():
        g = u[u.volt == v]
        for pole in ["cath", "anod"]:
            t, p = stats.ttest_rel(g[f"s_{pole}"], g[f"e_{pole}"])
            rows.append(dict(level=L, volt=v, pole=pole, n=len(g), start=g[f"s_{pole}"].mean(), end=g[f"e_{pole}"].mean(),
                             change=(g[f"e_{pole}"] - g[f"s_{pole}"]).mean(), p=p))
    return pd.DataFrame(rows)

def fig2a_stable(mapping, reading="experiment", mode="b", sets_sel="levels"):
    b = stability("FV", "baseline", mapping, reading, mode, sets_sel)
    q = stability("FV", "pulse", mapping, reading, mode, sets_sel)
    qh = q[q.level >= 1000]
    out = dict(baseline_tests=len(b), baseline_sig=int((b.p < 0.05).sum()), baseline_max_abs_change=b.change.abs().max(),
               pulse_1000plus_tests=len(qh), pulse_1000plus_sig=int((qh.p < 0.05).sum()),
               pulse_1000plus_max_abs_change=qh.change.abs().max() if len(qh) else np.nan)
    out["reproduced"] = verdict(out["baseline_sig"] == 0, (out["pulse_1000plus_sig"] == 0) if len(qh) else None)
    return out

sm = pd.DataFrame([dict(mapping=mapping, cells=reading, start_end=mode, sets=sets_sel, **fig2a_stable(mapping, reading, mode, sets_sel))
                   for mapping in ["top", "bottom"] for reading in READINGS for mode in ["a", "b"] for sets_sel in ["levels", "all"]])
print(sm.round(2).to_string(index=False))

In [ ]:
print("Main reading (top mapping, experiment means, first 10 against last 10), baseline per set and pole:")
print(stability("FV", "baseline", "top", "experiment", "b").round(3).to_string(index=False))
print("\nTop mapping, experiment means, first 10 against last 10, pulse images at 1000 to 1400 V/cm:")
q = stability("FV", "pulse", "top", "experiment", "b")
print(q[q.level >= 1000].round(3).to_string(index=False))
b19 = stability("FV", "baseline", "top", "experiment", "b", "all")
print(f"\nAll 19 sets, baseline, experiment means, 10-image windows: {int((b19.p < 0.05).sum())} of {len(b19)} tests at p < 0.05"
      f" (about {0.05 * len(b19):.1f} expected by chance); changes {b19.change.min():.2f} to {b19.change.max():.2f}%,"
      f" {int((b19.change < 0).sum())} of {len(b19)} negative")

**Result: not reproduced (under the assumed factor).** In the baseline images (top mapping,
experiment means, sets at the paper's levels, 10-image windows), the cathodic pole changes by
-3.34% at 780 V/cm (2-tailed paired t-test p = 0.018, n = 5) and by -2.82% at 1000 V/cm (p =
0.003). The other 14 baseline changes lie between -3.04% and +0.64% with p from 0.055 to 0.93.
Across readings, 1 to 4 of 16 baseline tests at the paper's levels and 4 to 9 of 38 over all
19 sets are significant. With experiment means over all sets, 4 of 38 are, about 1.9 being
expected by chance, and 32 of 38 changes are negative (-3.34 to +0.64%), a small bleaching
drift. In the pulse images at 1000 to 1400 V/cm none of the 6 tests differs with experiment
means (changes -0.78 to -2.01%, p 0.09 to 0.54), and 0 or 1 of 6 under the other readings. The
criterion counts any significant baseline change. In the sweep it is met only at 3.5 V/cm per volt.

## fig4a-fluovolt-limits

> FluoVolt measured and calculated IMP matched only at 43 and 150 V/cm; with stronger 1-us
> pulses the pole dF peaked at 71 +- 10% (cathodic) and -83.5 +- 5% (anodic), about +260 and
> -330 mV.

Collapsed pole dF per cell and level, mean across units; the maximum is the largest mean |dF|
over the levels at each pole. Measured IMP (dF / 26 x 100 mV) against the cell's pole IMPC by
2-tailed paired and unpaired t-tests at p < 0.05.

In [ ]:
def imp_vs_impc(dye, mapping, reading, background=True):
    df = (D if background else D_raw)
    df = df[df.dye == dye].copy()
    k = K[mapping]
    df["imp_cath"] = df.dF_cath / pef.SENS[dye] * 100
    df["imp_anod"] = df.dF_anod / pef.SENS[dye] * 100
    df["impc_cath"] = pef.impc_mV(df.volt * k, df.R_area_um, 0)
    df["impc_anod"] = -df.impc_cath
    u = units(df, reading)
    lv = pef.nearest_sets(df.volt, k)
    rows = []
    for L, v in lv.items():
        g = u[u.volt == v]
        for pole in ["cath", "anod"]:
            meas, calc = g[f"imp_{pole}"], g[f"impc_{pole}"]
            rows.append(dict(level=L, volt=v, pole=pole, n=len(g), dF=g[f"dF_{pole}"].mean(), dF_se=g[f"dF_{pole}"].sem(),
                             imp=meas.mean(), impc=calc.mean(), ratio=meas.mean() / calc.mean(),
                             p_paired=stats.ttest_rel(meas, calc).pvalue, p_unpaired=stats.ttest_ind(meas, calc).pvalue))
    return pd.DataFrame(rows)

def limits_check(t, dye, exp_c, tol_c, exp_a, tol_a):
    c = t[t.pole == "cath"].set_index("level")
    a = t[t.pole == "anod"].set_index("level")
    return dict(max_cath=c.dF.max(), at=int(c.dF.idxmax()), max_cath_ok=bool(abs(c.dF.max() - exp_c) <= tol_c),
                min_anod=a.dF.min(), at_a=int(a.dF.idxmin()), min_anod_ok=bool(abs(a.dF.min() - exp_a) <= tol_a))

def fig4a_limits(mapping, reading="experiment", tests=("p_paired", "p_unpaired")):
    t = imp_vs_impc("FV", mapping, reading)
    r = limits_check(t, "FV", 71, 10, -83.5, 5)
    for test in tests:
        low = t[t.level.isin([43, 150])]
        high = t[t.level >= 230]
        r[f"{test}_no_diff_43_150"] = bool((low[test] >= 0.05).all()) if len(low) else None
        below = high.assign(b=(high[test] < 0.05) & (high.imp.abs() < high.impc.abs()))
        r[f"{test}_below_each_230plus"] = bool(below.groupby("level").b.any().all()) if len(high) else None
        r[f"reproduced_{test}"] = verdict(r["max_cath_ok"], r["min_anod_ok"], r[f"{test}_no_diff_43_150"], r[f"{test}_below_each_230plus"])
    return r

print(pd.DataFrame([dict(mapping=mapping, cells=reading, **fig4a_limits(mapping, reading))
                    for mapping in ["top", "bottom"] for reading in READINGS]).round(1).to_string(index=False))
print("\nPer level, experiment means:")
for mapping in ["top", "bottom"]:
    print(mapping)
    print(imp_vs_impc("FV", mapping, "experiment").round(3).to_string(index=False))

**Result: not reproduced.** Under the top mapping the largest mean cathodic dF is 35.9% (at 570
V/cm; 34.4 and 35.7 under the other cell readings), against 71 +- 10, and the most negative
anodic value is -49.5% (1200 V/cm; -49.9 and -51.1), against -83.5 +- 5. Under the bottom
variant they are 31.7 to 35.8% and -48.8 to -51.1%, and no factor in the sweep meets them. Under
the top mapping there is no 43 V/cm set; at 150 V/cm measured IMP is already below IMPC (ratios
0.10 cathodic and 0.20 anodic, paired p < 0.001 and 0.001), so the match there fails. Measured IMP is below IMPC
at every level from 230 V/cm (ratios 0.06 to 0.25).

## fig4a-fluovolt-cathodic-decline

> FluoVolt cathodic polarization fell from 71% at 360 V/cm to 45 to 50% at 1000 to 1400 V/cm.

Collapsed cathodic pole dF, mean across units, at 360 V/cm (71 +- 10%) and at each of 1000, 1200
and 1400 V/cm (40.5 to 55%).

In [ ]:
def fig4a_decline(mapping, reading="experiment"):
    t = imp_vs_impc("FV", mapping, reading)
    c = t[t.pole == "cath"].set_index("level").dF
    r = dict(cath_360=c.get(360, np.nan), ok_360=bool(abs(c[360] - 71) <= 10) if 360 in c.index else None)
    for L in (1000, 1200, 1400):
        r[f"cath_{L}"] = c.get(L, np.nan)
        r[f"ok_{L}"] = bool(40.5 <= c[L] <= 55) if L in c.index else None
    r["reproduced"] = verdict(r["ok_360"], r["ok_1000"], r["ok_1200"], r["ok_1400"])
    return r

print(pd.DataFrame([dict(mapping=mapping, cells=reading, **fig4a_decline(mapping, reading))
                    for mapping in ["top", "bottom"] for reading in READINGS]).round(1).to_string(index=False))
c_all = units(D[D.dye == "FV"], "experiment").groupby("volt").dF_cath.mean()
print(f"Largest mean cathodic dF of any set: {c_all.max():.1f}% at {c_all.idxmax()} V (61% is the lower bound at 360 V/cm)")

**Result: not reproduced.** The collapsed cathodic dF at 360 V/cm is 23.5% under the top mapping
(20.1 and 24.1 under the other cell readings) and 25.5 to 27.7% under the bottom variant, against
71 +- 10. At 1000, 1200 and 1400 V/cm it is 28.9, 25.5 and 21.2% (experiment means, mean 25.2),
below the range 40.5 to 55%. The largest mean cathodic dF of any set is 36.1%, below the 61%
lower bound, so no factor can meet the criterion.

## Di-8-ANEPPS claims at 570 V/cm and above

Under the adopted `top` mapping the sets reach 1400 V/cm. The paper's 43 V/cm level has no set,
and 1200 V/cm is the 200 V set (1120 V/cm). Under the `bottom` variant the sets end at 430 V/cm
and these criteria cannot be tested; its values at the levels present are shown where they
exist. Pole dF at all 19 sets is in the data check.

### fig2b-di8-climbs-to-570

> Di-8-ANEPPS dF kept increasing with field strength up to 570 V/cm at both poles.

Per-image pole dF averaged over the 60 pulse images, per cell, then across units; |dF| must
increase at every step from 43 to 570 V/cm.

In [ ]:
def fig2b_climbs(mapping, reading="experiment", background=True):
    t = show_levels("Di8", ["pi_cath", "pi_anod"], background, readings=[reading], mappings=[mapping]).set_index("level").sort_index()
    t = t[t.index <= 570]
    steps = {pole: np.diff(np.abs(t[f"pi_{pole}"].values)) for pole in ["cath", "anod"]}
    up = bool(all((s > 0).all() for s in steps.values()))
    return dict(levels=",".join(map(str, t.index)), has_43=43 in t.index, has_570=570 in t.index,
                cath=" ".join(f"{x:.1f}" for x in t.pi_cath), anod=" ".join(f"{x:.1f}" for x in t.pi_anod),
                every_step_up=up, reproduced=verdict(up, True if 570 in t.index else None))

print(pd.DataFrame([dict(bg=bgname, mapping=mapping, cells=reading, **fig2b_climbs(mapping, reading, background))
                    for bgname, background in [("b", True), ("a", False)]
                    for mapping in ["top", "bottom"] for reading in READINGS]).to_string(index=False))

**Result: partially reproduced (under the assumed factor).** Under the top mapping, with
experiment means, both poles rise at every step from 150 to 570 V/cm (cathodic -1.6, 2.7, 6.6,
13.0%; anodic -6.2, -7.7, -10.6, -16.7%), and so they do over all cells and without background
subtraction. With the cell nearest the centre the anodic pole falls from 150 to 230 V/cm (-5.7 to
-4.5%). The step from 43 V/cm cannot be tested, as that level has no set. In the sweep the
criterion holds at every factor from 2 to 8 V/cm per volt; below 2 there is no 570 V/cm set.

### fig2b-di8-decline-within-set

> With Di-8-ANEPPS, responses to pulses of 780 V/cm and above decreased gradually within each
> set of 60 images.

Mean |dF| at the start and end of the pulse images, mean across units; start and end (a) first
and last image, (b) first 10 and last 10; poles (c) cathodic, (d) both. The 2-tailed paired
t-test per set.

In [ ]:
def fig2b_decline_rows(mapping, reading, mode):
    q = stability("Di8", "pulse", mapping, reading, mode)
    q = q[q.level >= 780].copy()
    q["abs_start"], q["abs_end"] = q.start.abs(), q.end.abs()
    q["end_below_start"] = q.abs_end < q.abs_start
    return q

def fig2b_decline(mapping, reading="experiment", mode="b", poles=("cath",)):
    q = fig2b_decline_rows(mapping, reading, mode)
    q = q[q.pole.isin(poles)]
    return dict(n_sets=q.level.nunique(), reproduced=bool(q.end_below_start.all()) if q.level.nunique() == 4 else None)

dd = pd.concat([fig2b_decline_rows("top", reading, mode).assign(cells=reading, start_end=mode)
                for reading in READINGS for mode in ["a", "b"]])
print("Top mapping:")
print(dd[["cells", "start_end", "level", "volt", "pole", "abs_start", "abs_end", "end_below_start", "p"]].round(3).to_string(index=False))
print("\nEnd below start in every set from 780 V/cm, by reading:")
print(dd.groupby(["cells", "start_end", "pole"]).end_below_start.all().unstack().to_string())

In [ ]:
# Control for drift: the same start-to-end change in every set, split into the offset shared by
# both poles ((cath + anod) / 2) and the polarization amplitude ((cath - anod) / 2). Top mapping,
# experiment means, first 10 against last 10 pulse images.
df = D[D.dye == "Di8"].copy()
for pole in ["cath", "anod"]:
    se = np.array([start_end(f, pef.N_BASE, 100, "b") for f in df[f"frames_{pole}"]])
    df[f"s_{pole}"], df[f"e_{pole}"] = se[:, 0], se[:, 1]
u = units(df, "experiment")
u["d_cath"] = u.e_cath.abs() - u.s_cath.abs()
u["d_anod"] = u.e_anod.abs() - u.s_anod.abs()
u["d_shared"] = (u.e_cath + u.e_anod) / 2 - (u.s_cath + u.s_anod) / 2
u["d_polar"] = (u.e_cath - u.e_anod) / 2 - (u.s_cath - u.s_anod) / 2
drift = u.groupby("volt")[["d_cath", "d_anod", "d_shared", "d_polar"]].mean()
drift.insert(0, "V/cm top", (drift.index * K["top"]).round())
drift["p_cath"] = [stats.ttest_1samp(g.d_cath, 0).pvalue for _, g in u.groupby("volt")]
print("Change from the first 10 to the last 10 pulse images (%; |dF| change at each pole, shared offset, polarization amplitude):")
print(drift.round(3).to_string())
lowset = drift[drift["V/cm top"] < 780]
print(f"\nSets below 780 V/cm (top): cathodic |dF| falls in {int((lowset.d_cath < 0).sum())} of {len(lowset)}, "
      f"mean change {lowset.d_cath.mean():.2f}%; shared offset mean change {lowset.d_shared.mean():.2f}%")
hiset = drift[drift["V/cm top"] >= 780]
print(f"Sets from 780 V/cm (top): cathodic |dF| falls in {int((hiset.d_cath < 0).sum())} of {len(hiset)}, mean change {hiset.d_cath.mean():.2f}%;"
      f" shared offset {hiset.d_shared.mean():.2f}%, polarization amplitude {hiset.d_polar.mean():.2f}%")

**Result: reproduced (under the assumed factor and pole).** Under the top mapping the cathodic
|dF| at the end of the pulse images is below the start in every set from 780 V/cm, under every
cell reading and both windows. With experiment means and 10-image windows it goes from 19.8 to
17.1, 22.2 to 19.7, 24.4 to 20.8 and 27.0 to 22.4% (p = 0.034, 0.069, 0.037, 0.142). The anodic
pole does not decline in any of these sets with experiment means and 10-image windows, and in
at most one set under the other readings, so reading (d), both poles, fails. Reading (c), the cathodic pole, is adopted. The text does not name the pole, and the paper
places the damage at the cathodic pole (Results 3.1, the cathodic pole dims at 780 V/cm; Results
3.4, the reduction is more pronounced at the cathodic pole). The drift control limits what this shows.
Below 780 V/cm the cathodic |dF| also falls within the set in 9 of 11 sets (mean -1.76%). From
780 V/cm it falls by -3.32% on average, of which the offset shared by both poles is -2.15% and the
polarization amplitude -1.17%. Most of the decline is a drift present at every field strength,
and the part specific to strong fields is small.

### fig4b-di8-limits

> With Di-8-ANEPPS and 1-us pulses, measured IMP fell below the calculated value at 570 V/cm,
> where dF peaked at 57 +- 14% (cathodic) and -55 +- 4% (anodic), over 600 mV.

In [ ]:
def fig4b_limits(mapping, reading="experiment", tests=("p_paired", "p_unpaired")):
    t = imp_vs_impc("Di8", mapping, reading)
    r = limits_check(t, "Di8", 57, 14, -55, 4)
    r["peaks_at_570"] = bool(r["at"] == 570 and r["at_a"] == 570) if 570 in t.level.values else None
    for test in tests:
        at570 = t[t.level == 570]
        r[f"{test}_below_at_570"] = bool(((at570[test] < 0.05) & (at570.imp.abs() < at570.impc.abs())).any()) if len(at570) else None
        low = t[t.level <= 360]
        r[f"{test}_not_below_43_360"] = bool(not ((low[test] < 0.05) & (low.imp.abs() < low.impc.abs())).any())
        r[f"reproduced_{test}"] = verdict(r["max_cath_ok"], r["min_anod_ok"], r["peaks_at_570"], r[f"{test}_below_at_570"], r[f"{test}_not_below_43_360"])
    return r

print(pd.DataFrame([dict(mapping=mapping, cells=reading, **fig4b_limits(mapping, reading))
                    for mapping in ["top", "bottom"] for reading in READINGS]).round(1).to_string(index=False))
print("\nPer level, experiment means:")
for mapping in ["top", "bottom"]:
    print(mapping)
    print(imp_vs_impc("Di8", mapping, "experiment").round(3).to_string(index=False))

**Result: not reproduced.** Under the top mapping Di-8-ANEPPS pole dF is largest at 1400 V/cm
(24.3% and -29.2%, experiment means), not at 570 V/cm (13.0% and -16.7%), against 57 +- 14 and
-55 +- 4. Measured IMP is below IMPC at every level, including 150 to 360 V/cm (ratios -0.14 to
0.54), so the second condition fails as well. The largest Di-8-ANEPPS means of any set are 24.3%
and -29.2%, so no factor meets the magnitudes.

### fig4b-first-10-pulses

> Averaged over only the first 10 of the 60 pulses, Di-8-ANEPPS cathodic polarization reached
> 66 +- 15% (710 +- 200 mV), so the 600-mV maximum is not the dye's limit.

In [ ]:
def fig4b_first10(mapping, reading="experiment"):
    t = show_levels("Di8", ["f10_cath", "dF_cath"], readings=[reading], mappings=[mapping]).set_index("level")
    ok = bool(abs(t.f10_cath.max() - 66) <= 15)
    return dict(max_first10=t.f10_cath.max(), at=int(t.f10_cath.idxmax()), within_66_15=ok, max_60=t.dF_cath.max(),
                above_60_max=bool(t.f10_cath.max() > t.dF_cath.max()), reproduced=verdict(ok, bool(t.f10_cath.max() > t.dF_cath.max())))

print(pd.DataFrame([dict(mapping=mapping, cells=reading, **fig4b_first10(mapping, reading))
                    for mapping in ["top", "bottom"] for reading in READINGS]).round(1).to_string(index=False))
f10 = units(D[D.dye == "Di8"], "experiment").groupby("volt").f10_cath.mean()
print(f"Over all 19 sets, experiment means: largest first-10 cathodic dF {f10.max():.1f}% at {f10.idxmax()} V")

**Result: not reproduced.** Over the first 10 pulse images the largest mean cathodic dF is 27.0%
(top mapping, 1400 V/cm, experiment means; 21.0 for the cell nearest the centre), against 66 +- 15.
It is above the 60-image maximum (24.3%) under every reading. The largest first-10 value of any
set is 27.0%, so no factor meets the criterion.

### fig4b-cathodic-asymmetry

> Above 780 V/cm the pole IMP fell progressively with field strength, more at the cathodic pole;
> at 1400 V/cm the cathodic pole no longer charged and the anodic pole fell to about -30% dF.

In [ ]:
def fig4b_asym(mapping, reading="experiment"):
    u = units(D[D.dye == "Di8"], reading)
    lv = pef.nearest_sets(u.volt, K[mapping])
    if not all(L in lv for L in (780, 1000, 1200, 1400)):
        return dict(reproduced=None)
    m = {L: u[u.volt == lv[L]] for L in (780, 1000, 1200, 1400)}
    c = {L: g.dF_cath.mean() for L, g in m.items()}
    a = {L: g.dF_anod.mean() for L, g in m.items()}
    dec = all(abs(c[L]) < abs(c[780]) for L in (1000, 1200, 1400)) and all(abs(a[L]) < abs(a[780]) for L in (1000, 1200, 1400))
    lost_c = 1 - abs(c[1400]) / abs(c[780])
    lost_a = 1 - abs(a[1400]) / abs(a[780])
    t1 = stats.ttest_1samp(m[1400].dF_cath, 0)
    r = dict(cath=" ".join(f"{c[L]:.1f}" for L in (780, 1000, 1200, 1400)), anod=" ".join(f"{a[L]:.1f}" for L in (780, 1000, 1200, 1400)),
             decreases=dec, lost_cath=lost_c, lost_anod=lost_a, more_at_cath=lost_c > lost_a, cath_1400=c[1400],
             p_cath_1400_vs_0=t1.pvalue, anod_1400=a[1400], anod_ok=bool(abs(a[1400] + 30) <= 3))
    r["reproduced"] = verdict(bool(dec), bool(lost_c > lost_a), bool(t1.pvalue >= 0.05), r["anod_ok"])
    return r

print("Levels 780, 1000, 1200, 1400 V/cm:")
print(pd.DataFrame([dict(mapping=mapping, cells=reading, **fig4b_asym(mapping, reading))
                    for mapping in ["top", "bottom"] for reading in READINGS]).round(3).to_string(index=False))

**Result: not reproduced (under the assumed factor).** Under the top mapping both poles keep
growing from 780 to 1400 V/cm (cathodic 18.1, 20.4, 22.5, 24.3%; anodic -21.2, -25.7, -27.8,
-29.2%, experiment means), so condition (1) fails under every cell reading. The cathodic value at
1400 V/cm differs from zero (one-sample p < 0.001; p = 0.053 for the cell nearest the centre), and
the anodic -29.2% is within -30 +- 3. No factor from 5 to 8 meets the criterion; below 5 there is
no 1400 V/cm set.

### fig4c-equator-follows-cosine

> Membrane regions nearer the equator were unaffected by pole electroporation and followed the
> cosine dependence of the Schwan equation up to 1000 V/cm.

Measured IMP (dF / 9.1%) against IMPC, averaged per unit within arcs beside the two equator
points (90 and 270 degrees from the cathode), (a) 15 and (b) 30 degrees wide, on the cathodic and
anodic side of each; 2-tailed paired t-test at 570, 780 and 1000 V/cm.

In [ ]:
def equator_arcs(width):
    th = np.arange(360)
    return {"90_cath_side": (th >= 90 - width) & (th < 90), "90_anod_side": (th > 90) & (th <= 90 + width),
            "270_anod_side": (th >= 270 - width) & (th < 270), "270_cath_side": (th > 270) & (th <= 270 + width)}

def fig4c_rows(mapping, reading, width):
    df = D[D.dye == "Di8"]
    lv = pef.nearest_sets(df.volt, K[mapping])
    rows = []
    for L in (570, 780, 1000):
        if L not in lv:
            continue
        g = df[df.volt == lv[L]]
        for name, sel in equator_arcs(width).items():
            meas = np.array([p[sel].mean() for p in g.profile]) / pef.SENS["Di8"] * 100
            calc = np.array([pef.impc_mV(lv[L] * K[mapping], r, np.arange(360))[sel].mean() for r in g.R_area_um])
            x = g.assign(meas=meas, calc=calc)
            x = units(x, reading) if reading != "experiment" else x.groupby("subject", as_index=False)[["meas", "calc"]].mean()
            p = stats.ttest_rel(x.meas, x.calc).pvalue
            rows.append(dict(cells=reading, width=width, level=L, arc=name, n=len(x), meas=x.meas.mean(), calc=x.calc.mean(),
                             ratio=x.meas.mean() / x.calc.mean(), p=p, differs=p < 0.05))
    return pd.DataFrame(rows)

def fig4c(mapping, reading="experiment", width=15):
    t = fig4c_rows(mapping, reading, width)
    if t.empty or t.level.nunique() < 3:
        return dict(n_differ=int(t.differs.sum()) if len(t) else 0, reproduced=verdict(False) if len(t) and t.differs.any() else None)
    return dict(n_differ=int(t.differs.sum()), n_tests=len(t), reproduced=not bool(t.differs.any()))

eq = pd.concat([fig4c_rows("top", reading, width) for reading in READINGS for width in (15, 30)])
print("Top mapping:")
print(eq.round(3).to_string(index=False))
print("\nArcs that differ, by reading:", eq.groupby(["cells", "width"]).differs.sum().to_dict())

**Result: not reproduced (under the assumed factor).** Under the top mapping, measured IMP in the
arcs beside the equator is below IMPC at 570, 780 and 1000 V/cm in most arcs (ratios -0.15 to 0.82
over readings). Of 12 arc-level tests per reading, 9 to 11 differ at p < 0.05 with 15-degree arcs
and all 12 with 30-degree arcs. The arcs on the cathode side of 90 degrees read about zero or
negative (ratios -0.13, 0.10 and 0.04 with experiment means and 15-degree arcs), where IMPC is
positive. No factor in the sweep meets the criterion.

### fig1-pole-dimming

> With Di-8-ANEPPS and 1-us pulses, polarization grows with field strength up to 570 V/cm; at
> 780 V/cm the cathodic pole stops increasing and dims, and at 1000 to 1400 V/cm the poles show
> reduced or absent polarization while regions nearer the equator stay polarized and may
> polarize more than the poles.

(1) The mean cathodic pole dF at 780 V/cm is not above that at 570 V/cm; (2) at 1400 V/cm the
largest mean dF of the cathode-facing hemisphere lies outside the +-5 degree polar arc.
Backgrounds (a) and (b).

In [ ]:
def fig1(mapping, reading="experiment", df0=None):
    df0 = D if df0 is None else df0
    u = units(df0[df0.dye == "Di8"], reading)
    lv = pef.nearest_sets(u.volt, K[mapping])
    if not all(L in lv for L in (570, 780, 1400)):
        return dict(reproduced=None)
    c570 = u[u.volt == lv[570]].dF_cath.mean()
    c780 = u[u.volt == lv[780]].dF_cath.mean()
    g14 = units(df0[(df0.dye == "Di8") & (df0.volt == lv[1400])], "centre" if reading == "centre" else "cells")
    if reading == "experiment":
        prof = np.mean([np.mean(np.stack(g.profile.values), 0) for _, g in g14.groupby("subject")], 0)
    else:
        prof = np.mean(np.stack(g14.profile.values), 0)
    hemi = np.r_[np.arange(270, 360), np.arange(0, 91)]
    peak = hemi[np.argmax(prof[hemi])]
    off = (peak + 180) % 360 - 180
    return dict(cath_570=c570, cath_780=c780, cond1=bool(c780 <= c570), peak_deg_from_cathode=off, peak_dF=prof[peak],
                pole_dF=pef.arc(prof, 0), cond2=bool(abs(off) > 5), reproduced=verdict(bool(c780 <= c570), bool(abs(off) > 5)))

print(pd.DataFrame([dict(bg=bgname, mapping=mapping, cells=reading, **fig1(mapping, reading, df0))
                    for bgname, df0 in [("b", D), ("a", D_raw)] for mapping in ["top", "bottom"] for reading in READINGS]).round(2).to_string(index=False))

**Result: not reproduced (under the assumed factor).** Under the top mapping the mean cathodic
dF at 780 V/cm (18.1%) is above that at 570 V/cm (13.0%), so condition (1) fails under every cell
and background reading (10.6 against 12.1% for the cell nearest the centre; 7.4 against 9.9%
without background). At 1400 V/cm the peak of the cathode-facing hemisphere lies 12 to 46 degrees
from the pole, so condition (2) holds. No factor from 5 to 8 meets the criterion; below 5 there
is no 1400 V/cm set.

### fig7b-charging-during-pulse

> Normalized to 150 V/cm, pole polarization during the conditioning pulse fell 1.5 times below
> linear scaling at 560 V/cm and 2-fold at 760 V/cm, and further at stronger fields (p < 0.01 for
> every point from 560 V/cm).

Collapsed pole dF as % of the 150 V/cm value, against the line 100% x E / 150. Normalization
(a) per cell, (b) of the means; pole (c) each, (d) mean of the poles. One-sample t-test of the
per-unit values against the line.

In [ ]:
def fig7b_rows(mapping, reading):
    u = units(D[D.dye == "Di8"], reading)
    lv = pef.nearest_sets(u.volt, K[mapping])
    if not all(L in lv for L in (150, 570, 780, 1000, 1200, 1400)):
        return pd.DataFrame()
    base = u[u.volt == lv[150]].set_index("unit")
    rows = []
    for pole in ["cath", "anod", "mean"]:
        def val(g):
            return (g.dF_cath - g.dF_anod) / 2 if pole == "mean" else g[f"dF_{pole}"].abs()
        b = val(base)
        r = dict(mapping=mapping, cells=reading, pole=pole)
        okp = True
        for L in (570, 780, 1000, 1200, 1400):
            g = u[u.volt == lv[L]].set_index("unit")
            line = 100 * lv[L] / lv[150]
            pc = 100 * val(g) / b
            r[f"ratio_a_{L}"] = line / pc.mean()
            r[f"ratio_b_{L}"] = line / (100 * val(g).mean() / b.mean())
            p = stats.ttest_1samp(pc, line).pvalue
            r[f"p_{L}"] = p
            okp &= bool(p < 0.01 and pc.mean() < line)
        r["all_below_p01"] = okp
        for nm in ("a", "b"):
            r[f"ok_{nm}"] = bool(abs(r[f"ratio_{nm}_570"] - 1.5) <= 0.15 and abs(r[f"ratio_{nm}_780"] - 2) <= 0.2
                                 and all(r[f"ratio_{nm}_{L}"] > 2 for L in (1000, 1200, 1400)) and okp)
        rows.append(r)
    return pd.DataFrame(rows)

def fig7b(mapping, reading="experiment"):
    t = fig7b_rows(mapping, reading)
    return dict(reproduced=bool((t.ok_a | t.ok_b).any()) if len(t) else None)

f7 = pd.concat([fig7b_rows("top", reading) for reading in READINGS])
print("Top mapping (560 V/cm is the 100 V set, 760 V/cm the 140 V set at 784 V/cm):")
print(f7.round(3).to_string(index=False))

**Result: not reproduced (under the assumed factor).** Under the top mapping, normalized to the
150 V/cm set (25 V), no reading meets the criterion. At 25 V both poles share a bleaching offset
(mean of the poles -3.9% in Di-8-ANEPPS, data check), which makes the anodic |dF| at 150 V/cm
too large and the cathodic one too small, and so biases the anodic ratios up and the cathodic
ones down. The anodic pole with experiment means gives linear-over-measured ratios of 1.46 at
560 V/cm (within 1.5 +- 0.15) and 1.62 at 760 V/cm (outside 2 +- 0.2), and 1.72, 1.76 and 2.08
at 1000 to 1400 V/cm, so it exceeds 2 only at 1400. Every datapoint of that reading is below the
line at p < 0.01. Its values are nearest the paper's, but it is the reading the offset affects
most. The cathodic ratios are 0.63 to 0.94 with experiment means. The mean of the poles, in which
the offset cancels, gives 0.55 to 0.85 with experiment means, so the measured polarization lies
above the linear line at every level. With per-cell normalization over all cells or for the
centre cell, the mean of the poles gives ratios such as -10.7 and 16.9, because the 150 V/cm
values are near zero there; those ratios carry no information. In the sweep only 5.6 and 6.25
V/cm per volt have every level, and neither meets the criterion.

## fig5b-conductivity-calibration

> By Eq. (2), the effective membrane conductivity barely affects charging while it stays far
> below cytosol and medium conductivities; a 10-fold rise reduces IMP by about 2% and a 100-fold
> rise by 12%.

Eq. (2) with the constants of Methods 2.3 (lambda_i 0.4 S/m, lambda_o 1.64 S/m, R 8 um, d 5 nm,
intact lambda_m 3e-7 S/m); no recordings. Reduction read as (a) 1 - f/1.5 and (b) 1 - f(lambda_m)
/ f(3e-7 S/m).

In [ ]:
f0 = pef.f_factor(3e-7)
print(f"f at the intact 3e-7 S/m: {f0:.5f} (1.5 for a non-conducting membrane)")
rows = []
for fold in [1, 3, 10, 30, 100, 300, 1000]:
    f = pef.f_factor(3e-7 * fold)
    rows.append(dict(fold=fold, lambda_m=3e-7 * fold, f=f, reduction_a=100 * (1 - f / 1.5), reduction_b=100 * (1 - f / f0)))
t5 = pd.DataFrame(rows)
print(t5.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
for fold, e, tol in [(10, 2, 0.2), (100, 12, 1.2)]:
    r = t5.set_index("fold").loc[fold]
    print(f"{fold}-fold: (a) {r.reduction_a:.2f}%, (b) {r.reduction_b:.2f}%; expected {e} +- {tol}%;"
          f" within: (a) {abs(r.reduction_a - e) <= tol}, (b) {abs(r.reduction_b - e) <= tol}")
fold_for_2 = [fo for fo in np.logspace(0, 3, 3001) if 100 * (1 - pef.f_factor(3e-7 * fo) / 1.5) >= 2][0]
print(f"A 2% reduction, reading (a), needs a {fold_for_2:.1f}-fold rise")

**Result: partially reproduced.** Eq. (2) with the constants of Methods 2.3 gives f = 1.498 for the
intact membrane. A 10-fold rise in conductivity reduces IMP by 1.33% under reading (a) and 1.20%
under (b), against 2 +- 0.2; it takes a 15.2-fold rise to reach 2%. A 100-fold rise reduces IMP
by 11.87% and 11.75%, within 12 +- 1.2. The claim's qualitative point holds: the reduction stays
small until the conductivity rises more than 30-fold (3.9% at 30-fold).

## Outcomes across the factor from volts to V/cm

The factor is not recorded, so each criterion is run over factors from 1.5 to 8 V/cm per volt:
the two endpoint readings (1.72 and 5.6), the two-dimensional rod estimates for the paper's
geometry (4.64 and 7.19, data check) and steps between. Sets are named by the paper's levels as
elsewhere (nearest within 15%). Each criterion is run on its adopted reading: experiment means,
background subtracted, 10-image windows, the paired test, the cathodic pole for
fig2b-di8-decline-within-set, and every smooth for fig3-fluovolt-nonlinear. "yes" is reproduced, "no" not reproduced,
"-" a level the criterion needs has no set at that factor.

In [ ]:
SWEEP = [1.5, 1.72, 2, 2.5, 3, 3.5, 4, 4.64, 5, 5.6, 6.25, 7.19, 8]
show = {True: "yes", False: "no", None: "-"}

def sweep_row(k):
    name = f"k{k:g}"
    K[name] = k
    lv = pef.nearest_sets(D.volt, k)
    f3 = fig3_row(name)
    return {"factor": k, "V/cm": f"{25 * k:.0f}-{250 * k:.0f}", "levels with a set": ",".join(str(L) for L in lv),
            "FV slope": round(f3["FV_slope"], 1), "Di8 slope": round(f3["Di8_slope"], 1),
            "fig3-dye-sensitivity": f3["reproduced"],
            "fig3-fluovolt-nonlinear": nonlinear_verdict(nonlinear_rows(name)[0]),
            "fig2a-fluovolt-saturation": fig2a_sat(name)["reproduced"],
            "fig2a-fluovolt-stable": fig2a_stable(name)["reproduced"],
            "fig4a-fluovolt-limits": fig4a_limits(name, tests=("p_paired",))["reproduced_p_paired"],
            "fig4a-fluovolt-cathodic-decline": fig4a_decline(name)["reproduced"],
            "fig2b-di8-climbs-to-570": fig2b_climbs(name)["reproduced"],
            "fig2b-di8-decline-within-set": fig2b_decline(name)["reproduced"],
            "fig4b-di8-limits": fig4b_limits(name, tests=("p_paired",))["reproduced_p_paired"],
            "fig4b-first-10-pulses": fig4b_first10(name)["reproduced"],
            "fig4b-cathodic-asymmetry": fig4b_asym(name)["reproduced"],
            "fig4c-equator-follows-cosine": fig4c(name)["reproduced"],
            "fig1-pole-dimming": fig1(name)["reproduced"],
            "fig7b-charging-during-pulse": fig7b(name)["reproduced"]}

def sweep(DD):
    # Every criterion at every factor, on the pole table DD (the criterion functions read the global D).
    global D
    D_main = D
    D = DD
    try:
        return pd.DataFrame([sweep_row(k) for k in SWEEP]).set_index("factor")
    finally:
        D = D_main

sw = sweep(D)
crit = [c for c in sw.columns if c.startswith("fig")]
print(sw[["V/cm", "levels with a set", "FV slope", "Di8 slope"]].to_string())
print()
print(sw[crit].apply(lambda col: col.map(show)).T.to_string())
print("\nFactors at which each criterion is reproduced:", {c: [k for k in SWEEP if sw.loc[k, c] is True] for c in crit})
sw_e = sweep(D_edge)
print("\nEdge trace (variant), slopes and outcomes:")
print(sw_e[["FV slope", "Di8 slope"]].T.to_string())
print(sw_e[crit].apply(lambda col: col.map(show)).T.to_string())
diff = {c: [k for k in SWEEP if sw.loc[k, c] != sw_e.loc[k, c]] for c in crit}
print("\nFactors at which the edge trace changes a result:", {c: v for c, v in diff.items() if v})

With the peak trace, no factor run from 1.5 to 8 reproduces fig3-dye-sensitivity (nor any on
the finer grid from 1.40 to 2.30 in its section),
fig3-fluovolt-nonlinear, the FluoVolt and Di-8-ANEPPS maxima (fig2a-fluovolt-saturation,
fig4a-fluovolt-limits, fig4a-fluovolt-cathodic-decline, fig4b-di8-limits,
fig4b-first-10-pulses), fig4b-cathodic-asymmetry, fig4c-equator-follows-cosine,
fig1-pole-dimming or fig7b-charging-during-pulse. fig2b-di8-climbs-to-570 holds at every factor
from 2 to 8, fig2b-di8-decline-within-set at every factor from 5.6 to 8 (below 5.6 there is no
1400 V/cm set), and fig2a-fluovolt-stable only at 3.5. At the geometric factors (4.64 to 7.19)
the slopes are 4.4 to 8.0 (FluoVolt) and 1.6 to 3.2 (Di-8-ANEPPS). With the edge trace the
results differ only for fig2a-fluovolt-stable (met at 8, not at 3.5) and
fig4c-equator-follows-cosine at 2 (tested and not met). At the adopted 5.6, every result is the
same under both traces.

## Summary

| Claim | Outcome |
|---|---|
| fig3-dye-sensitivity | not reproduced (assumed factor) |
| fig3-fluovolt-nonlinear | not reproduced (assumed factor) |
| fig2a-fluovolt-saturation | not reproduced |
| fig2a-fluovolt-stable | not reproduced (assumed factor) |
| fig4a-fluovolt-limits | not reproduced |
| fig4a-fluovolt-cathodic-decline | not reproduced |
| fig2b-di8-climbs-to-570 | partially reproduced (assumed factor) |
| fig2b-di8-decline-within-set | reproduced (assumed factor and pole) |
| fig4b-di8-limits | not reproduced |
| fig4b-first-10-pulses | not reproduced |
| fig4b-cathodic-asymmetry | not reproduced (assumed factor) |
| fig4c-equator-follows-cosine | not reproduced (assumed factor) |
| fig1-pole-dimming | not reproduced (assumed factor) |
| fig7b-charging-during-pulse | not reproduced (assumed factor) |
| fig5b-conductivity-calibration | partially reproduced |
| fig2cd-stable-before-electroporation, fig2c-1000-within-set, fig2d-lasting-reduction | blocked: data missing |
| fig4d-short-pulses-same-limit | blocked: data missing |
| fig5a-gd-extends-anodic, fig5a-gd-linear, fig5a-gd-cathodic, fig5a-gd-differences, fig5c-gd-reduces-conductivity | blocked: data missing |
| fig6a-ca-single-pulse, fig6b-ca-train-responders, fig6b-ca-650 | blocked: data missing |
| fig7ab-test-pulse-recovery, fig7c-fast-conductivity-drop, fig7c-full-reseal-mild | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The files give the pulse in volts. The factor to V/cm is taken as 5.6 V/cm per volt (1400 V/cm
  at 250 V). This is an assumption, supported by the electrode geometry of Methods 2.2 (4.64 to
  7.19 in a two-dimensional model). The `bottom` reading (1.72) is shown as a variant, and every
  criterion is run over factors from 1.5 to 8. Sets are named by the paper's level nearest them.
  The sweep separates two kinds of failure. The FluoVolt and Di-8-ANEPPS maxima fail at every
  factor. Only fig2b-di8-climbs-to-570 (2 to 8), fig2b-di8-decline-within-set (5.6 to 8) and
  fig2a-fluovolt-stable (3.5 only) hold at some factors.
- The paper analysed one cell per experiment; the files show 3 to 6 cells per field and do not
  say which. Experiment means, the cell nearest the image centre, and all cells are each run.
- Membrane traces are found automatically, not drawn by hand. The peak trace is drawn inwards
  in some cells, and one object in c7 is two touching cells. Moving the trace within 3 pixels,
  or following the cell edge, does not bring the mean pole dF near the paper's values, and at
  the adopted factor the edge trace changes no result. Dark frames are left out, so most
  baselines have 39 images. The pixel size (0.267 um) is assumed from the camera and objective,
  and tau is from reference [20].
- The mean of per-image pole dF and the collapsed-image pole dF are the same number here,
  because both use the same baseline mean as F0, so the Fig. 2 and Fig. 4 measures coincide.

What was found about the dataset:

- It holds ten 19-set escalations of 1-us pulses (25 to 250 V), five per dye, from one day. It
  holds none of the paper's alternating, 0.3-us, Gd3+, Ca2+ or two-pulse experiments. Its
  voltages span a factor of 10, so no single factor maps them onto the paper's 43 to 1400 V/cm
  series.
- The files state an electrode distance of 1.00 mm; the paper states 1.6 mm. They give no field
  strength, field direction, pixel size or cell identity. The field direction is clear from the
  data (cathode-facing side towards the bottom of the image in every experiment).
- Frame 0 is dark in 175 of 190 sets, and 51 other frames are dark, which the files do not
  mention.
- No recording is deposited twice.